In [1]:
import subprocess, time, requests

def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

In [2]:
%%writefile spam_app.py
# Spam detection API with an optional Redis cache
import os
import time
import joblib
import redis
from fastapi import FastAPI, HTTPException, Response
from pydantic import BaseModel

MODEL_PATH = os.environ.get("MODEL_PATH", "data/model.joblib")
REDIS_HOST = os.environ.get("REDIS_HOST")  # if not set, then API runs without cache
CACHE_TTL = int(os.environ.get("CACHE_TTL", "3600"))

app = FastAPI(title="Spam Detection API")
_model = None
cache = redis.Redis(host=REDIS_HOST, port=6379, decode_responses=True) if REDIS_HOST else None


@app.on_event("startup")
def load_model():
    global _model
    _model = joblib.load(MODEL_PATH)
    print(f"loaded model from {MODEL_PATH}, cache host: {REDIS_HOST}", flush=True)


class PredictRequest(BaseModel):
    text: str


@app.get("/healthz")
def healthz():
    if _model is None:
        raise HTTPException(status_code=503, detail="model not loaded")
    return {"status": "ok"}


@app.post("/predict")
def predict(request: PredictRequest, response: Response):
    if _model is None:
        raise HTTPException(status_code=503, detail="model not loaded")

    t0 = time.perf_counter()
    key = f"pred:{request.text}"
    label = cache.get(key) if cache else None

    if label is not None:
        status = "HIT"
    else:
        label = _model.predict([request.text]).tolist()[0]
        if cache:
            cache.set(key, label, ex=CACHE_TTL)
        status = "MISS"

    elapsed_ms = (time.perf_counter() - t0) * 1000
    response.headers["X-Cache"] = status
    response.headers["X-Process-Time-ms"] = f"{elapsed_ms:.3f}"
    print(f"cache {status} ({elapsed_ms:.3f} ms): {request.text[:40]!r}", flush=True)
    return {"label": label}

Overwriting spam_app.py


In [3]:
%%writefile requirements-predictor.txt
fastapi
uvicorn[standard]
scikit-learn
joblib
pydantic
redis

Overwriting requirements-predictor.txt


In [4]:
%%writefile docker-compose.yml
services:
  api:
    build:
      context: .
      dockerfile: Dockerfile.multistage
    image: spam-api:cache
    ports:
      - "8080:8080"
    environment:
      REDIS_HOST: cache
      CACHE_TTL: "3600"
    depends_on:
      - cache

  cache:
    image: redis:7-alpine

Writing docker-compose.yml


In [5]:
!docker compose up -d --build

[+] Running 0/1
 ⠋ cache Pulling                                                           0.1s 
[+] Running 0/1
 ⠙ cache Pulling                                                           0.2s 
[+] Running 0/1
 ⠹ cache Pulling                                                           0.3s 
[+] Running 0/1
 ⠸ cache Pulling                                                           0.4s 
[+] Running 0/1
 ⠼ cache Pulling                                                           0.5s 
[+] Running 0/1
 ⠴ cache Pulling                                                           0.6s 
[+] Running 0/1
 ⠦ cache Pulling                                                           0.7s 
[+] Running 0/1
 ⠧ cache Pulling                                                           0.8s 
[+] Running 0/1
 ⠇ cache Pulling                                                           0.9s 
[+] Running 0/1
 ⠏ cache Pulling                                                           1.0s 
[+] Running 0/1
 ⠋ cache Pulli

In [6]:
sh("docker compose ps");

NAME         IMAGE            COMMAND                  SERVICE   CREATED         STATUS         PORTS
a2-api-1     spam-api:cache   "uvicorn spam_app:ap…"   api       3 minutes ago   Up 3 minutes   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
a2-cache-1   redis:7-alpine   "docker-entrypoint.s…"   cache     3 minutes ago   Up 3 minutes   6379/tcp



In [7]:
time.sleep(3)
URL = "http://localhost:8080/predict"

for text in ["Congratulations! You have WON a laptop. Claim NOW at bit.ly/xyz123",
             "Can you send me the notes from basketball class?"]:
    for attempt in [1, 2]:
        r = requests.post(URL, json={"text": text})
        print(f"call {attempt}: {r.json()}  X-Cache={r.headers['X-Cache']}  "
              f"server={r.headers['X-Process-Time-ms']} ms")
    print()

call 1: {'label': 'spam'}  X-Cache=MISS  server=12.856 ms
call 2: {'label': 'spam'}  X-Cache=HIT  server=0.296 ms

call 1: {'label': 'ham'}  X-Cache=MISS  server=1.443 ms
call 2: {'label': 'ham'}  X-Cache=HIT  server=0.173 ms



In [9]:
import statistics

def timed_post(text):
    t0 = time.perf_counter()
    r = requests.post(URL, json={"text": text})
    client_ms = (time.perf_counter() - t0) * 1000
    return r.headers["X-Cache"], float(r.headers["X-Process-Time-ms"]), client_ms, r.json()["label"]

run = int(time.time())
results = {"MISS": {"server": [], "client": []}, "HIT": {"server": [], "client": []}}
mismatches = 0

for i in range(50):
    if i % 2 == 0:
        text = f"WIN a FREE gift card now! Click here: win-now.co/claim {run}-{i}"
    else:
        text = f"Running a bit late for lunch, be there in 10 min {run}-{i}"

    status1, server1, client1, label1 = timed_post(text)
    status2, server2, client2, label2 = timed_post(text)
    assert status1 == "MISS" and status2 == "HIT"
    mismatches += label1 != label2

    results["MISS"]["server"].append(server1)
    results["MISS"]["client"].append(client1)
    results["HIT"]["server"].append(server2)
    results["HIT"]["client"].append(client2)

print(f"label mismatches between miss and hit: {mismatches}\n")
print(f"{'':6}{'server median (ms)':>20}{'client median (ms)':>20}")
for status in ["MISS", "HIT"]:
    s = statistics.median(results[status]["server"])
    c = statistics.median(results[status]["client"])
    print(f"{status:6}{s:>20.3f}{c:>20.3f}")

speedup = statistics.median(results["MISS"]["server"]) / statistics.median(results["HIT"]["server"])
print("\n")
print(f"server side speedup on cache hit: {speedup:.1f}x")

label mismatches between miss and hit: 0

        server median (ms)  client median (ms)
MISS                 1.050               2.505
HIT                  0.187               1.653


server side speedup on cache hit: 5.6x


In [10]:
sh("docker compose logs api --tail 10")
sh("docker compose exec cache redis-cli dbsize")
sh("docker compose exec cache redis-cli --scan --count 5 | head -3");

api-1  | cache HIT (0.179 ms): 'Running a bit late for lunch, be there i'
api-1  | INFO:     172.18.0.1:59788 - "POST /predict HTTP/1.1" 200 OK
api-1  | cache MISS (4.790 ms): 'WIN a FREE gift card now! Click here: wi'
api-1  | INFO:     172.18.0.1:59798 - "POST /predict HTTP/1.1" 200 OK
api-1  | cache HIT (0.349 ms): 'WIN a FREE gift card now! Click here: wi'
api-1  | INFO:     172.18.0.1:59812 - "POST /predict HTTP/1.1" 200 OK
api-1  | cache MISS (0.988 ms): 'Running a bit late for lunch, be there i'
api-1  | INFO:     172.18.0.1:59820 - "POST /predict HTTP/1.1" 200 OK
api-1  | cache HIT (0.278 ms): 'Running a bit late for lunch, be there i'
api-1  | INFO:     172.18.0.1:59836 - "POST /predict HTTP/1.1" 200 OK

102

pred:Running a bit late for lunch, be there in 10 min 1789762370-37
pred:WIN a FREE gift card now! Click here: win-now.co/claim 1789762323-38
pred:WIN a FREE gift card now! Click here: win-now.co/claim 1789762370-42

